In [1]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import roc_curve
from sklearn.metrics import matthews_corrcoef
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [3]:

df = pd.read_csv('/content/archive.zip')
print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (6607, 20)
   Hours_Studied  Attendance Parental_Involvement Access_to_Resources  \
0             23          84                  Low                High   
1             19          64                  Low              Medium   
2             24          98               Medium              Medium   
3             29          89                  Low              Medium   
4             19          92               Medium              Medium   

  Extracurricular_Activities  Sleep_Hours  Previous_Scores Motivation_Level  \
0                         No            7               73              Low   
1                         No            8               59              Low   
2                        Yes            7               91           Medium   
3                        Yes            8               98           Medium   
4                        Yes            6               65           Medium   

  Internet_Access  Tutoring_Sessions Family_Income Teacher_Q

In [4]:
print(df.columns)
print(df.isnull().sum())

Index(['Hours_Studied', 'Attendance', 'Parental_Involvement',
       'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours',
       'Previous_Scores', 'Motivation_Level', 'Internet_Access',
       'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type',
       'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities',
       'Parental_Education_Level', 'Distance_from_Home', 'Gender',
       'Exam_Score'],
      dtype='object')
Hours_Studied                  0
Attendance                     0
Parental_Involvement           0
Access_to_Resources            0
Extracurricular_Activities     0
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               0
Internet_Access                0
Tutoring_Sessions              0
Family_Income                  0
Teacher_Quality               78
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_

In [5]:
df = df[['Hours_Studied', 'Attendance',
         'Previous_Scores',
         'Sleep_Hours', 'Parental_Education_Level',
         'School_Type','Family_Income',
         'Exam_Score']]

print(df.head())

   Hours_Studied  Attendance  Previous_Scores  Sleep_Hours  \
0             23          84               73            7   
1             19          64               59            8   
2             24          98               91            7   
3             29          89               98            8   
4             19          92               65            6   

  Parental_Education_Level School_Type Family_Income  Exam_Score  
0              High School      Public           Low          67  
1                  College      Public        Medium          61  
2             Postgraduate      Public        Medium          74  
3              High School      Public        Medium          71  
4                  College      Public        Medium          70  


In [6]:
X = df[['Hours_Studied', 'Attendance',
        'Previous_Scores', 'Family_Income',
        'Sleep_Hours', 'Parental_Education_Level',
        'School_Type']]

y = df['Exam_Score']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
X = pd.get_dummies(X, drop_first=True, dtype=int)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [10]:
y_pred = model.predict(X_test)

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)

Mean Absolute Error (MAE): 1.2996022810039005
Root Mean Squared Error (RMSE): 2.253963419010059


In [12]:
import gradio as gr
import pandas as pd

model_columns = X.columns

def predict_exam_score(hours, attendance, previous_score, sleep_hours, family_income, parental_education, school_type):
    new_student = pd.DataFrame([{
        'Hours_Studied': hours,
        'Attendance': attendance,
        'Previous_Scores': previous_score,
        'Sleep_Hours': sleep_hours,
        'Family_Income': family_income,
        'Parental_Education_Level': parental_education,
        'School_Type': school_type
    }])

    new_student_encoded = pd.get_dummies(new_student, dtype=int).reindex(columns=model_columns, fill_value=0)
    score = round(float(model.predict(new_student_encoded)[0]), 2)

    if score >= 85:
        grade, color, msg = "A+ (Excellent)", "#10b981", "Top tier academic trajectory!"
    elif score >= 70:
        grade, color, msg = "B (Good)", "#3b82f6", "Solid performance. A bit more study time can push this to an A."
    elif score >= 55:
        grade, color, msg = "C (Average)", "#f59e0b", "Passing, but attendance or study hours need a boost."
    else:
        grade, color, msg = "D (At Risk)", "#ef4444", "High risk. Immediate academic intervention recommended."

    html_output = f"""
    <div style="background: linear-gradient(145deg, #1e1b4b, #0f172a); border: 1px solid #6366f1; border-radius: 16px; padding: 28px; text-align: center; box-shadow: 0 10px 25px rgba(99, 102, 241, 0.25);">
        <p style="color: #94a3b8; font-size: 14px; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 8px;">Predicted Final Exam Score</p>
        <h1 style="font-size: 56px; font-weight: 800; margin: 0; background: linear-gradient(90deg, #818cf8, #38bdf8); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
            {score} <span style="font-size: 24px; -webkit-text-fill-color: #64748b;">/ 100</span>
        </h1>
        <div style="display: inline-block; background: {color}22; color: {color}; border: 1px solid {color}; padding: 6px 16px; border-radius: 999px; font-weight: 600; font-size: 14px; margin-top: 14px;">
            Grade: {grade}
        </div>
        <p style="color: #cbd5e1; font-size: 14px; margin-top: 16px; margin-bottom: 0;">{msg}</p>
    </div>
    """
    return html_output

custom_css = """
body, .gradio-container { background-color: #090d16 !important; font-family: 'Inter', sans-serif; }
.header-banner { background: linear-gradient(90deg, #312e81 0%, #1e1b4b 50%, #0f172a 100%); border: 1px solid #4338ca; border-radius: 16px; padding: 24px; text-align: center; margin-bottom: 20px; }
.header-banner h1 { color: #ffffff; font-size: 32px; font-weight: 800; margin: 0; letter-spacing: -0.5px; }
.card-box { background: #111827 !important; border: 1px solid #1f2937 !important; border-radius: 16px !important; padding: 18px !important; }
#predict-btn { background: linear-gradient(90deg, #6366f1, #0ea5e9) !important; border: none !important; color: white !important; font-weight: 700 !important; font-size: 16px !important; padding: 12px !important; border-radius: 12px !important; transition: transform 0.15s ease; }
#predict-btn:hover { transform: scale(1.02); box-shadow: 0 0 20px rgba(99, 102, 241, 0.5); }
footer { display: none !important; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Base(primary_hue="indigo", neutral_hue="slate")) as demo:
    gr.HTML("""
        <div class="header-banner">
            <h1>Student Score Predictor</h1>
        </div>
    """)

    with gr.Row():
        # Left Column: Academic Metrics
        with gr.Column(scale=1, elem_classes="card-box"):
            gr.Markdown("### 📚 Academic Habits")
            hours = gr.Slider(0, 45, value=23, step=1, label="Hours Studied (per week)")
            attendance = gr.Slider(50, 100, value=85, step=1, label="Attendance (%)")
            previous_score = gr.Slider(40, 100, value=75, step=1, label="Previous Exam Score")
            sleep_hours = gr.Slider(4, 12, value=7, step=1, label="Sleep Hours (per night)")

        # Middle Column: Socioeconomic & School Factors
        with gr.Column(scale=1, elem_classes="card-box"):
            gr.Markdown("### 🏫 Background & Environment")
            family_income = gr.Radio(["Low", "Medium", "High"], value="Medium", label="Family Income")
            parental_education = gr.Dropdown(["High School", "College", "Postgraduate"], value="College", label="Parental Education Level")
            school_type = gr.Radio(["Public", "Private"], value="Public", label="School Type")

        # Right Column: Prediction Output Card
        with gr.Column(scale=1):
            predict_btn = gr.Button("✨ Predict Exam Score", elem_id="predict-btn")
            output_card = gr.HTML(value=predict_exam_score(23, 85, 75, 7, "Medium", "College", "Public"))

    inputs_list = [hours, attendance, previous_score, sleep_hours, family_income, parental_education, school_type]
    predict_btn.click(fn=predict_exam_score, inputs=inputs_list, outputs=output_card)
    for inp in inputs_list:
        inp.change(fn=predict_exam_score, inputs=inputs_list, outputs=output_card)

demo.launch(share=True)


/tmp/ipykernel_970/709703435.py:53: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Base(primary_hue="indigo", neutral_hue="slate")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b3bcd713f36dbcc78.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:

# Decision Tree Regressor
tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

print("Decision Tree MAE:", mean_absolute_error(y_test, tree_pred))
print("Decision Tree RMSE:", np.sqrt(mean_squared_error(y_test, tree_pred)))


# Predict exam score for a new student
print("\n--- Student Score Predictor ---")

hours = float(input("Enter hours studied: "))
attendance = float(input("Enter attendance percentage: "))
previous_score = float(input("Enter previous exam score: "))
sleep_hours = float(input("Enter sleep hours: "))

Family_Income = input("Family Income (Low/Medium/High): ")
parental_education = input("Parents education level (High School/College/Graduate): ")
school_type = input("School type (Public/Private): ")


new_student = pd.DataFrame([{
    'Hours_Studied': hours,
    'Attendance': attendance,
    'Previous_Scores': previous_score,
    'Sleep_Hours': sleep_hours,
    'Family_Income': Family_Income,
    'Parental_Education_Level': parental_education,
    'School_Type': school_type,

}])

# Apply same encoding as training data
new_student = pd.get_dummies(
    new_student,
    drop_first=True,
    dtype=int
)

new_student = new_student.reindex(
    columns=X.columns,
    fill_value=0
)

prediction = model.predict(new_student)

print("Predicted exam score:", round(prediction[0], 2))


Decision Tree MAE: 2.1210287443267775
Decision Tree RMSE: 3.776469867429986

--- Student Score Predictor ---
Enter hours studied: 8
Enter attendance percentage: 92
Enter previous exam score: 89
Enter sleep hours: 8
Family Income (Low/Medium/High): Medium
Parents education level (High School/College/Graduate): Graduate
School type (Public/Private): public
Predicted exam score: 67.62
